# **AfriLink SDK — End-to-End Finetune Demo**

This notebook walks through the **complete workflow** of the AfriLink SDK:

| Step | What | API |
|------|------|-----|
| 1 | Install SDK | `pip install afrilink-sdk` |
| 2 | Authenticate (DataSpires + CINECA) | `client.authenticate()` |
| 3 | Browse available models & datasets | `client.list_available_models()` |
| 4 | Prepare dataset | pandas DataFrame |
| 5 | Submit finetune job | `client.finetune(...).run(wait=True)` |
| 6 | Download trained weights | `client.download_model(job_id, dir)` |
| 7 | Load adapter & run local inference | `PeftModel.from_pretrained(base, dir)` |
| 8 | Recover session (if cert expires) | `client.recover_session()` |
| 9 | Route inference via HuggingFace | `client.inference(prompt, model_id)` |
| 10 | Convert to GGUF & run locally | `llama.cpp` + `ollama` |
| 11 | Publish to HuggingFace Hub | `huggingface_hub` |

**Prerequisites:** A free DataSpires account — sign up at https://dataspires.com

You can explore how it works with the example input or rerun the cells with your own input.

---
## 1. **Install the AfriLink SDK**

AfriLink SDK gives you one-line access to GPUs, models and datasets, all ready to use directly from a notebook. Authenticate, submit LoRA finetune jobs, download trained weights, and run inference — all without leaving your notebook.

Just run `pip install afrilink-sdk` and follow the rest of the steps below!

### Built-in User Guide

Once installed, a full reference manual is accessible via forward-slash commands directly in your notebook — no browser required.

In [ ]:
!pip install -q afrilink-sdk

import afrilink
print(f"AfriLink SDK v{afrilink.__version__} ready")

CINECA credentials provisioned successfully
AfriLink SDK v0.5.9 ready


In [ ]:
# Full topic index — shows all available help pages
afrilink/help

In [ ]:
# Quick-start cheat sheet
afrilink/quickstart

In [ ]:
# CINECA A100 hardware specs (nvidia-smi style)
afrilink/specs

In [ ]:
# Any topic is one slash away — try some others:
# afrilink/finetune   — full finetune() parameter reference
# afrilink/training   — training modes (low / medium / high)
# afrilink/recover    — session recovery workflow
# afrilink/inference  — HuggingFace endpoint routing
# afrilink/billing    — credits, invoices & vouchers
afrilink/finetune

---
## 2. **Authenticate with your DataSpires Account**

A single `client.authenticate()` call handles everything:

1. **DataSpires** — you'll be prompted for your email & password (for logging usage data)

After this cell you have an SSH-authenticated session to High Performance Compute with tried and tested links to models and data.

In [ ]:
from afrilink import AfriLinkClient

client = AfriLinkClient()
client.authenticate()  # Interactive: prompts for DataSpires credentials


DataSpires Authentication Required
Your DataSpires account is used for usage tracking and billing.
Don't have an account? Sign up at https://dataspires.com

DataSpires Email: ianomunga@gmail.com
DataSpires Password: ··········

Authenticating with DataSpires...
  Authenticated as: ianomunga@gmail.com
  User ID: ccab4bb3-2705-4979-be14-2ca8bf871fbb
  Balance: $8.57

------------------------------------------------------------

Authenticating with CINECA...
(Using org-wide shared credentials)
Starting headless authentication...
Installing Smallstep CLI...
Extracting...
Installed to /root/.local/bin/step
Bootstrapping Smallstep CA...
CA bootstrapped successfully
Installing Selenium...
Installing Google Chrome...
[1/4] Pre-launching headless browser...
      Browser ready.
[2/4] Starting step ssh login...
      [step] ✔ Provisioner: cineca-hpc (OIDC) [client: step-ca]
      [step] Your default web browser has been opened to visit:
      [step] https://sso.hpc.cineca.it/realms/CINECA-HPC/p

CinecaAuthResult(success=True, cert_path='ssh-agent', key_path='ssh-agent', expires_at=None, username='iomunga0', error=None)

In [ ]:
# Quick sanity check for checking that the high performance compute is linked well :)
code, out, err = client.run_command("hostname && echo $WORK")
print(out)

login05.leonardo.local
/leonardo_work/AIH4A_dataspire



---
## 3. **Browse AfriLink's Available Models & Datasets**

The SDK ships with a registry of pre-configured models (text LLMs and vision-language models) and datasets.

For this demo, the model that's ready for full testing is the `qwen2.5-0.5b` Multimodal model.

In [ ]:
from afrilink import list_models, list_datasets

models = list_models()

print("TEXT MODELS")
print("-" * 65)
for m in models:
    if m["type"] == "text":
        print(f"  {m['id']:20} {m['parameters_b']:.2f}B  {m['min_gpu_memory_gb']:>2}GB VRAM  {m['name']}")

print("\nVISION-LANGUAGE MODELS")
print("-" * 65)
for m in models:
    if m["type"] == "vision":
        print(f"  {m['id']:20} {m['parameters_b']:.2f}B  {m['min_gpu_memory_gb']:>2}GB VRAM  {m['name']}")

print("\nDATASETS")
print("-" * 65)
for d in list_datasets():
    n = d.get('num_examples') or 'N/A'
    print(f"  {d['id']:25} {str(n):>8} examples  {d['name']}")

TEXT MODELS
-----------------------------------------------------------------
  qwen2.5-0.5b         0.50B   4GB VRAM  Qwen 2.5 0.5B
  gemma-3-270m         0.27B   2GB VRAM  Gemma 3 270M
  ministral-3b         3.30B   8GB VRAM  Ministral 3B Reasoning
  llama-3.2-1b         1.00B   4GB VRAM  Llama 3.2 1B
  deepseek-r1-1.5b     1.50B   6GB VRAM  DeepSeek R1 Distill Qwen 1.5B

VISION-LANGUAGE MODELS
-----------------------------------------------------------------
  llava-1.5-7b         7.00B  16GB VRAM  LLaVA 1.5 7B
  moondream2           1.90B   8GB VRAM  Moondream 2
  florence-2-base      0.23B   4GB VRAM  Florence 2 Base
  smolvlm-256m         0.26B   2GB VRAM  SmolVLM 256M Instruct
  internvl2-1b         1.00B   4GB VRAM  InternVL2 1B

DATASETS
-----------------------------------------------------------------
  multilingual-thinking          N/A examples  Multilingual Thinking
  alpaca                       52000 examples  Stanford Alpaca
  dolly                        15000 examples

In [ ]:
# Check resource requirements for a specific model
client.get_model_requirements("qwen2.5-0.5b", "low")

{'model': 'qwen2.5-0.5b',
 'model_type': 'text',
 'training_mode': 'low',
 'recommended_gpus': 1,
 'min_memory_gb': 4,
 'parameters_b': 0.5,
 'context_length': 32768,
 'requires_vision_encoder': False}

---
## 4. **Prepare a Small Example Dataset**

The SDK accepts all these kinds of dataset configurations:
- **pandas DataFrame** with a `text` column (Alpaca-style prompt+response in one string)
- **HuggingFace Dataset**
- **File path** to a local JSONL/CSV

Below we create a small example. In production you'd use a real dataset loaded from your notebook with `pandas`

In [ ]:
import pandas as pd

# Alpaca-style: each row is a self-contained prompt + response
rows = [
    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\nWhat is machine learning?\n\n"
    "### Response:\nMachine learning is a branch of artificial intelligence that "
    "enables systems to learn patterns from data and improve with experience.",

    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\nExplain gradient descent in one sentence.\n\n"
    "### Response:\nGradient descent is an optimisation algorithm that iteratively "
    "adjusts model parameters in the direction that minimises the loss function.",

    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\nWhat is a neural network?\n\n"
    "### Response:\nA neural network is a computing architecture composed of "
    "interconnected layers of nodes that learns to map inputs to outputs.",
]

dataset = pd.DataFrame({"text": rows})
print(f"Dataset: {len(dataset)} rows")
print(dataset.head())

Dataset: 3 rows
                                                text
0  Below is an instruction that describes a task....
1  Below is an instruction that describes a task....
2  Below is an instruction that describes a task....


---
## 5. **Submit Finetune Job to AfriLink's HPC**

`client.finetune()` creates the job; `.run(wait=True)` submits it to SLURM and blocks until completion.

Behind the scenes the SDK:
1. Serialises your DataFrame to JSONL and uploads it to a suitable compute node
2. Submits the finetune job and polls the compute node until the job finishes

The job specification pipeline needs values for these variables:

```
job = client.finetune(
    model="qwen2.5-0.5b",    # The model you choose to finetune
    training_mode="low",      # How much training: "low", "medium", or "high"
    data=data,                # Your dataset (DataFrame, HF Dataset, or file path)
    gpus=1,                   # Number of A100 GPUs to use
    time_limit="01:00:00",    # Maximum time your job should run for
    backend="cineca",         # HPC backend: "cineca" (default), "eversetech", "agh", or "acf"
)
```

In [ ]:
# Create and inspect the job (does not submit yet)
job = client.finetune(
    model="qwen2.5-0.5b",
    training_mode="low",     # QLoRA, 4-bit, rank 8
    data=dataset,
    gpus=1,
    time_limit="01:00:00",
)

cfg = job.spec.training_config
print(f"Job ID:       {job.job_id}")
print(f"Model:        {job.spec.model}")
print(f"Mode:         {job.spec.training_mode.value}")
print(f"GPUs:         {job.spec.gpus}")
print(f"LoRA rank:    {cfg.lora_r}")
print(f"Quantisation: {cfg.quantization_bits}-bit" if cfg.use_quantization else "Quantisation: off")
print(f"Batch size:   {cfg.batch_size}")
print(f"LR:           {cfg.learning_rate}")

Job ID:       46ef9be2
Model:        qwen2.5-0.5b
Mode:         low
GPUs:         1
LoRA rank:    8
Quantisation: 4-bit
Batch size:   2
LR:           0.0002


In [ ]:
# Submit to SLURM and wait for completion
result = job.run(wait=True, poll_interval=30)

print(f"\nResult: {result}")


Preparing finetune job: ft-qwen2.5-0.5b-46ef9be2
Uploading dataset (3 rows) to CINECA...
Uploading: /tmp/tmp7n1r01oa.jsonl -> $WORK/datasets/46ef9be2/train.jsonl
Upload complete: 0.00 MB in 4.4s
  Dataset: $WORK/datasets/46ef9be2/train.jsonl
  Container: $WORK/containers/afrilink-qwen2.5-0.5b.sif

Submitting to SLURM...
Job submitted: 34302214
  Name: ft-qwen2.5-0.5b-46ef9be2
  Model: qwen2.5-0.5b
  GPUs: 1
  Training mode: low

Waiting for job completion (timeout: 14400s)...

Job completed successfully!

Result: {'job_id': '46ef9be2', 'slurm_job_id': '34302214', 'status': 'completed', 'output_dir': '$WORK/finetune_outputs/46ef9be2', 'model_path': '$WORK/finetune_outputs/46ef9be2/', 'billing': {'total_gpu_minutes': 0.0, 'total_gpu_hours': 0.0, 'rate_per_gpu_hour': 2.0, 'total_cost_usd': 0.0}}


In [ ]:
# (Optional) Check logs while waiting or after completion
print(job.get_logs(tail=50))

  "learning_rate": 0.0002
}
Job Configuration:
{
  "model_path": "/workspace/models/qwen2.5-0.5b",
  "dataset_path": "/workspace/data/train.jsonl",
  "output_dir": "/workspace/output",
  "lora_r": 8,
  "lora_alpha": 16,
  "batch_size": 2,
  "gradient_accumulation_steps": 8,
  "num_epochs": 3,
  "learning_rate": 0.0002
}

Loading model from: /workspace/models/qwen2.5-0.5b
trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184

Loading dataset from: /workspace/data/train.jsonl

Starting training...
{'train_runtime': 3.1527, 'train_samples_per_second': 2.855, 'train_steps_per_second': 0.952, 'train_loss': 0.5028878847757975, 'epoch': 3.0}

Saving model to: /workspace/output

Training complete!

Listing output files:
total 29
drwxr-xr-x  5 iomunga0 interactive     4096 Feb 17 10:31 .
drwxr-xr-x 12 iomunga0 interactive     4096 Feb 17 10:31 ..
-rw-r--r--  1 iomunga0 interactive      683 Feb 17 10:31 adapter_config.json
-rw-r--r--  1 iomunga0 interactive  4350392 Feb 17

---
## 6. **Download Trained Model Weights after Finetune is done**

The adapter files (`adapter_model.safetensors`, `adapter_config.json`, tokenizer files) are downloaded flat into the target directory — ready for `PeftModel.from_pretrained()`.

Use `client.download_model(result["job_id"], MODEL_DIR)` with a pre-specified directory for where you'd like the model files to be saved in your notebook's file system.
The extra steps here are just to print out status logs.

In [ ]:
if result.get("status") != "completed":
    print(f"Job did not complete successfully (status: {result.get('status')})")
    print("Check logs with: job.get_logs()")
    if result.get("error"):
        print(f"\nError output:\n{result['error'][:500]}")
else:
    MODEL_DIR = f"./my-model-{result['job_id']}"

    client.download_model(result["job_id"], MODEL_DIR)

    # Verify downloaded files
    import os
    for f in sorted(os.listdir(MODEL_DIR)):
        size = os.path.getsize(os.path.join(MODEL_DIR, f))
        print(f"  {f:40} {size/1024:.1f} KB")

Download complete: 19.30 MB in 47.6s
Model ready at: ./my-model-46ef9be2
  Load with: PeftModel.from_pretrained(base_model, "./my-model-46ef9be2")
  README.md                                5.0 KB
  adapter_config.json                      0.7 KB
  adapter_model.safetensors                4248.4 KB
  added_tokens.json                        0.6 KB
  merges.txt                               1632.7 KB
  special_tokens_map.json                  0.6 KB
  tokenizer.json                           11154.5 KB
  tokenizer_config.json                    7.1 KB
  training_args.bin                        5.1 KB
  vocab.json                               2711.8 KB


---
## 7. **Load Adapter & Run Inference**

Load the base model, attach the LoRA adapter, and generate text. You can keep experimenting at this point, or finetune more models!

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# MODEL_DIR is set in cell 16 (only if job completed)
if 'MODEL_DIR' not in dir():
    raise RuntimeError("No model downloaded — the training job did not complete successfully.")

BASE_MODEL = "Qwen/Qwen2.5-0.5B"

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Attaching LoRA adapter...")
model = PeftModel.from_pretrained(base_model, MODEL_DIR)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print("Model loaded.")

Loading base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded.


In [ ]:
# Generate a response
prompt = (
    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\nWhat is transfer learning?\n\n"
    "### Response:\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        do_sample=True,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Below is an instruction that describes a task.

### Instruction:
What is transfer learning?

### Response:
Transfer learning is a machine learning technique that involves training a model on one dataset and then applying the learned knowledge to a new, related dataset.

In transfer learning, the model architecture of the original model is used to extract features from the new dataset, and then the learned features are used to make predictions on the new dataset.

Transfer learning can be used to improve the performance of a model on a new dataset by reducing the amount of data needed for training. This is because the model architecture is already optimized for the original dataset, and the new dataset is similar to the original dataset in terms of features.

However, transfer learning may also cause overfit


---
## Utility: **Queue Status & Job Management**

Come back to this code cell to check the list of finetune jobs you've put on AfriLink's processing queue using `client.list_jobs()`

In [ ]:
# List your SLURM jobs
for j in client.list_jobs():
    print(j)

# Cancel a job (uncomment and set ID):
# client.cancel_job("SLURM_JOB_ID")

# Run arbitrary commands on CINECA:
code, out, err = client.run_command("squeue -u $USER")
print(out)

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)



---
## 8. **Session Recovery (Past 12 Hours)**

SSH certificates expire after ~12 hours. The SDK watches this in the background and warns you at 60, 30, 15, and 5 minutes before expiry.

When the warning fires — or when you come back to a notebook the next day — call  to:

1. **Re-authenticate** with CINECA (fresh SSH cert, no credentials re-entry)
2. **Check all tracked SLURM jobs** and report their status
3. **Download completed models** automatically (if you pass a directory)
4. **Register email notifications** for jobs that are still running

Your SLURM jobs keep running on the cluster even after your certificate expires — you just need fresh credentials to check on them.

In [ ]:
# Recover session and download any completed models
recovery = client.recover_session("./recovered-models")

# Check what happened
print(f"Re-authenticated: {recovery.re_authenticated}")
print(f"Jobs found: {len(recovery.jobs)}")
print(f"Models downloaded: {len(recovery.files_retrieved)}")

# Inspect individual job statuses
for job_id, info in recovery.jobs.items():
    status = info["status"]
    print(f"  {job_id}: {status}")

If you just want to re-authenticate without downloading anything, call it without arguments:

In [ ]:
# Just re-authenticate (no download)
client.recover_session()

# Check how much time you have on the new certificate
print(f"Certificate valid for {client.cert_minutes_remaining:.0f} more minutes")

---
## 9. **Route Inference via HuggingFace Endpoints**

`client.inference()` lets you send prompts to **any HuggingFace model** — from the public Inference API to your own private endpoints — without leaving the notebook and without needing a running CINECA session.

This is useful for:
- **Testing your finetuned model** once you have uploaded it to the HF Hub
- **Comparing against a baseline** (e.g. the un-finetuned base model)
- **Production inference** via a private HF Inference Endpoint

A [HuggingFace token](https://huggingface.co/settings/tokens) is required for gated models (Llama 2, Gemma, Mistral-v0.2) and private models. Public open-access models work without a token, subject to rate limits.

In [ ]:
# Public model — no token needed, rate-limited to ~10 req/min without a token
result = client.inference(
    "Explain LoRA fine-tuning in one sentence.",
    model_id="HuggingFaceH4/zephyr-7b-beta",
)

if result.success:
    print(result.text)
else:
    print(f"Error {result.status_code}: {result.error}")

LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique that inserts small trainable rank-decomposition matrices into a frozen pre-trained model, reducing the number of trainable parameters by orders of magnitude while achieving comparable performance to full fine-tuning.


In [ ]:
# With a HuggingFace token + generation parameters
# Get a free token at: https://huggingface.co/settings/tokens
HF_TOKEN = "hf_..."  # replace with your token

result = client.inference(
    "What is transfer learning? Answer in one sentence.",
    model_id="mistralai/Mistral-7B-Instruct-v0.2",
    hf_token=HF_TOKEN,
    parameters={"max_new_tokens": 128, "temperature": 0.6, "return_full_text": False},
)

if result.success:
    print(result.text)
else:
    print(f"Error {result.status_code}: {result.error}")

Transfer learning is a technique where a model trained on a large dataset is adapted for a different but related task, reusing the learned representations to reduce training time and data requirements.


In [ ]:
# Test inference on your own finetuned model after uploading it to the HF Hub
# (See Step 11 below for how to upload)
#
# result = client.inference(
#     "What is machine learning?",
#     model_id="your-hf-username/my-finetuned-qwen",
#     hf_token=HF_TOKEN,   # needed if your repo is private
#     parameters={"max_new_tokens": 200},
# )
# print(result.text)

# Private endpoint (if you have deployed one on HuggingFace Inference Endpoints):
# result = client.inference(
#     payload={"inputs": "Hello!"},
#     endpoint_url="https://xyz.endpoints.huggingface.cloud",
#     hf_token=HF_TOKEN,
# )

# The full InferenceResult fields:
print(f"Status : {result.status_code}")
print(f"Success: {result.success}")
print(f"Text   : {result.text[:200]}")
# result.raw contains the full decoded JSON from HuggingFace

---
## 10. **Convert to GGUF & Run Locally with Ollama**

Once you have downloaded your finetuned adapter weights, you can **merge the adapter into the base model** and convert to GGUF format for offline inference with `ollama` or `llama.cpp` — no GPU required.

This is done entirely outside the AfriLink SDK using standard open-source tooling.

**What this section covers:**
1. Merge the LoRA adapter into the full base model weights
2. Convert to GGUF with `llama.cpp`
3. Create an Ollama Modelfile and run it locally

> **Note:** The conversion step requires enough RAM to hold the full model in fp16. For a 0.5B model that is well under 2 GB.

In [ ]:
# ── Step 1: Merge LoRA adapter into the full base model weights ────────────
#
# PeftModel.merge_and_unload() fuses the LoRA delta weights into the base model
# and returns a plain HuggingFace model ready for saving.

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch, os

BASE_MODEL = "Qwen/Qwen2.5-0.5B"
MERGED_DIR = "./my-model-merged"

print("Loading base model in fp16...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",   # CPU for merging avoids GPU OOM on small instances
)

print("Attaching and merging LoRA adapter...")
peft_model = PeftModel.from_pretrained(base, MODEL_DIR)
merged = peft_model.merge_and_unload()
merged.eval()

print(f"Saving merged model to {MERGED_DIR} ...")
merged.save_pretrained(MERGED_DIR)
AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(MERGED_DIR)
print("Merged model saved.")

Loading base model in fp16...
Attaching and merging LoRA adapter...
Saving merged model to ./my-model-merged ...
Merged model saved.


In [ ]:
# ── Step 2: Convert to GGUF using llama.cpp ───────────────────────
#
# llama.cpp ships a Python conversion script that handles most HF architectures
# including Qwen2, Llama, Mistral, Gemma, and others.

!git clone --depth 1 https://github.com/ggerganov/llama.cpp /tmp/llama.cpp 2>/dev/null || echo "already cloned"
!pip install -q -r /tmp/llama.cpp/requirements.txt

GGUF_FILE = "./my-model-merged/model-q4_k_m.gguf"

# Convert to f16 GGUF first
!python /tmp/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile ./my-model-merged/model-f16.gguf \
    --outtype f16

# Build llama-quantize
!cmake /tmp/llama.cpp -B /tmp/llama.cpp/build -DLLAMA_NATIVE=OFF -DCMAKE_BUILD_TYPE=Release 2>/dev/null
!cmake --build /tmp/llama.cpp/build --config Release -j$(nproc) --target llama-quantize 2>/dev/null

# Quantize to Q4_K_M (good balance of quality vs size)
!/tmp/llama.cpp/build/bin/llama-quantize \
    ./my-model-merged/model-f16.gguf \
    {GGUF_FILE} \
    Q4_K_M

!ls -lh {GGUF_FILE}

In [ ]:
# ── Step 3: Smoke test with llama-cpp-python ───────────────────────

!pip install -q llama-cpp-python

from llama_cpp import Llama

llm = Llama(model_path=GGUF_FILE, n_ctx=512, n_threads=4, verbose=False)

response = llm(
    "Below is an instruction.\n\n### Instruction:\nWhat is gradient descent?\n\n### Response:\n",
    max_tokens=128,
    stop=["\n\n"],
)
print(response["choices"][0]["text"])

Gradient descent is an optimisation algorithm that iteratively adjusts model parameters in the direction that minimises the loss function.


In [ ]:
# ── Step 4: Create an Ollama Modelfile and import locally ────────────
#
# Ollama reads a Modelfile that points at the GGUF and sets a system prompt.
# After "ollama create" you can chat via "ollama run my-model" from any terminal.
# Prerequisites: install Ollama from https://ollama.com

import os

gguf_abs = os.path.abspath(GGUF_FILE)
modelfile_content = (
    f"FROM {gguf_abs}\n\n"
    "PARAMETER temperature 0.7\n"
    "PARAMETER top_p 0.9\n"
    "PARAMETER stop \"\\n\\n\"\n\n"
    "SYSTEM \"\"\"\n"
    "You are a helpful AI assistant trained with AfriLink on CINECA Leonardo HPC.\n"
    "Answer questions clearly and concisely.\n"
    "\"\"\"\n"
)

modelfile_path = "./Modelfile"
with open(modelfile_path, "w") as fh:
    fh.write(modelfile_content)

print(f"Modelfile written to: {os.path.abspath(modelfile_path)}")
print("\nTo import and run with Ollama (run in your terminal):")
print(f"  ollama create my-afrilink-model -f {os.path.abspath(modelfile_path)}")
print( "  ollama run my-afrilink-model")

# Uncomment to run directly if Ollama is installed in this environment:
# !ollama create my-afrilink-model -f {modelfile_path}
# !ollama run my-afrilink-model "What is machine learning?"

---
## 11. **Publish Your Model to HuggingFace Hub**

Share your finetuned adapter (or the full merged model) publicly or privately on the HuggingFace Hub so it can be used with `client.inference()`, loaded by others, or served via HF Inference Endpoints.

This uses the standard `huggingface_hub` library — no AfriLink SDK involvement.

**You will need:**
- A free HuggingFace account at https://huggingface.co
- A write-access token from https://huggingface.co/settings/tokens

In [ ]:
# ── Option A: Push the LoRA adapter only (lightweight, ~5–20 MB) ─────────
#
# Recommended: small and fast to upload.
# Users load it on top of the base model with PeftModel.from_pretrained().

!pip install -q huggingface_hub

from huggingface_hub import HfApi

HF_TOKEN   = "hf_..."                                  # your HF write token
HF_REPO_ID = "your-username/my-finetuned-qwen-lora"    # created if it does not exist

api = HfApi(token=HF_TOKEN)

# Create the repo (set private=True to keep it private)
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, private=False)

# Upload the adapter directory
api.upload_folder(
    folder_path=MODEL_DIR,
    repo_id=HF_REPO_ID,
    commit_message="Upload LoRA adapter finetuned with AfriLink on CINECA A100",
)

print(f"Adapter uploaded to: https://huggingface.co/{HF_REPO_ID}")
print("\nLoad it anywhere with:")
print(f"  from peft import PeftModel")
print(f"  model = PeftModel.from_pretrained(base_model, '{HF_REPO_ID}')")

In [ ]:
# ── Option B: Push the full merged model (larger, ~1–14 GB) ───────────
#
# Use this for direct HF pipeline loading or HF Inference Endpoint deployment.

HF_REPO_MERGED = "your-username/my-finetuned-qwen-merged"

api.create_repo(repo_id=HF_REPO_MERGED, exist_ok=True, private=False)

api.upload_folder(
    folder_path=MERGED_DIR,
    repo_id=HF_REPO_MERGED,
    commit_message="Upload full merged model finetuned with AfriLink on CINECA A100",
)

print(f"Merged model uploaded to: https://huggingface.co/{HF_REPO_MERGED}")

In [ ]:
# ── Option C: Push the GGUF file (for llama.cpp / Ollama / LM Studio users) ─

HF_REPO_GGUF = "your-username/my-finetuned-qwen-gguf"

api.create_repo(repo_id=HF_REPO_GGUF, exist_ok=True, private=False)

api.upload_file(
    path_or_fileobj=GGUF_FILE,
    path_in_repo="model-q4_k_m.gguf",
    repo_id=HF_REPO_GGUF,
    commit_message="Upload Q4_K_M GGUF finetuned with AfriLink",
)

print(f"GGUF uploaded to: https://huggingface.co/{HF_REPO_GGUF}")
print("\nAnyone can now run it with Ollama:")
print(f"  ollama run hf.co/{HF_REPO_GGUF}")

In [ ]:
# ── Test inference on the uploaded model via AfriLink ────────────────
#
# Once the model is on the Hub and inference is enabled
# (may take a few minutes for the Inference API to warm up on first use):

result = client.inference(
    "What is machine learning?",
    model_id=HF_REPO_ID,        # the adapter repo you uploaded above
    hf_token=HF_TOKEN,          # needed if repo is private
    parameters={"max_new_tokens": 128, "return_full_text": False},
)

if result.success:
    print(result.text)
else:
    print(f"Status {result.status_code}: {result.error}")

---
## **Summary**

```python
pip install afrilink-sdk

import afrilink
afrilink/help                                                  # built-in user guide

from afrilink import AfriLinkClient

client = AfriLinkClient()
client.authenticate()                                          # DataSpires + CINECA

job = client.finetune(model="qwen2.5-0.5b", data=df, gpus=1)  # create job
result = job.run(wait=True)                                    # submit & wait

client.download_model(result["job_id"], "./my-model")          # download adapter

client.recover_session("./models")                             # re-auth + recover

# Local inference
model = PeftModel.from_pretrained(base, "./my-model")          # load adapter

# HuggingFace inference (no CINECA needed)
result = client.inference("Your prompt", model_id="org/model") # HF endpoint
print(result.text)

# Export & share
# merged = peft_model.merge_and_unload()    -> GGUF -> Ollama
# api.upload_folder(...)                    -> HuggingFace Hub
```

For the full docs, check out the platform at https://dataspires.com and sign up for an account to use AfriLink today!